# GPU Debugging with ROCgdb

## Abstract

<p style="text-align:justify;">In this lab, you will debug a HIP GPU workload using ROCgdb, AMD's heterogeneous debugger for CPU and GPU applications. You will compile a HIP application with debug symbols, debug host and device execution in a single session, inspect GPU wavefronts and lanes, step through kernel execution, inspect registers and device memory, and verify correct kernel output on your GPU hardware.</p>

## Objectives

After completing this lab, you will be able to:

- Prepare a HIP application for source-level GPU debugging by enabling debug symbols and targeting your GPU architecture
- Use ROCgdb to debug CPU and GPU code together, including wavefront and lane-level inspection
- Interpret debugging outputs such as threads, lanes, registers, memory, and breakpoints to analyze GPU execution behavior

## Introduction

<p style="text-align:justify;">The AMD ROCm software platform provides a comprehensive open-source ecosystem for GPU computing on AMD accelerators, enabling developers to build, debug, and optimize high-performance applications across heterogeneous CPU-GPU systems. ROCm supports HIP (Heterogeneous-compute Interface for Portability), allowing developers to write portable GPU code while maintaining familiar programming models. As GPU applications grow in complexity, effective debugging becomes essential for verifying correctness, understanding execution behavior, and diagnosing issues across both host and device code.</p>

<p style="text-align:justify;">ROCgdb extends the standard GNU Debugger (GDB) to support AMD GPUs, providing a unified debugging environment for CPU and GPU execution. It enables developers to debug HIP applications in a single session, inspect kernel execution, analyze runtime behavior, and examine low-level details such as registers, memory, and thread execution states. By leveraging ROCdbgapi, ROCgdb interacts directly with AMDGPU hardware, allowing fine-grained visibility into execution at the wavefront and lane level.</p>

### ROCgdb Unified CPU-GPU Debugging Workflow

<p style="text-align:justify;">The ROCgdb debugging workflow integrates CPU and GPU debugging into a seamless, unified experience. Understanding this workflow is essential for effective heterogeneous application debugging.</p>

<p style="text-align:justify;"> **Stage 1: HIP Application (CPU + GPU Kernel)** - The debugging process begins with a HIP application that contains both host (CPU) code and device (GPU) kernel code. This application must be compiled with debug symbols to enable source-level debugging.</p>

<p style="text-align:justify;"> **Stage 2: Compile with Debug** - The application is compiled using `hipcc` with the `-g` flag to include debug symbols and `-O0` to disable optimizations. This compilation step ensures that the debugger can accurately map GPU execution back to source code, enabling meaningful inspection of variables, registers, and execution flow.</p>

<p style="text-align:justify;">**Stage 3: Launch ROCgdb** - ROCgdb is launched with the compiled application as its target. The debugger loads both CPU and GPU debug information, preparing for unified debugging of heterogeneous code.</p>

<p style="text-align:justify;"> **Stage 4: CPU Debug and GPU Debug** - Once launched, ROCgdb provides simultaneous debugging capabilities for both CPU and GPU execution. On the CPU side, you can set breakpoints in host code, step through function calls, inspect host variables, and control program flow. On the GPU side, you can set breakpoints in device kernels, inspect wavefront and lane execution, examine GPU registers and device memory, and analyze parallel execution behavior.</p>

<p style="text-align:justify;"> **Stage 5: ROCgdb Backend** - The ROCgdb backend manages the debugging session, coordinating between CPU debugging (using standard GDB functionality) and GPU debugging (using AMD-specific extensions). This backend translates debugger commands into appropriate actions for both execution contexts.</p>

<p style="text-align:justify;"> **Stage 6: ROCdbgapi** - ROCdbgapi is the low-level API that provides the interface between ROCgdb and the GPU hardware. It handles device-specific debugging operations such as wavefront control, lane inspection, register access, and memory queries. This API abstracts hardware details while providing comprehensive debugging capabilities.</p>

<p style="text-align:justify;"> **Stage 7: GPU Hardware** - At the bottom of the stack is the physical AMD GPU hardware, which executes kernels and responds to debugging commands. ROCdbgapi communicates with the GPU driver to control execution, set breakpoints, and retrieve execution state.</p>

<p style="text-align:justify;">This lab introduces users to the ROCm debugging workflow through a practical, end-to-end debugging session of a HIP-based GPU workload. The first part of the lab focuses on environment readiness and debug build configuration. You will verify GPU availability, validate the ROCm debugging toolchain, and compile a HIP application with debug symbols and appropriate architecture targets to enable accurate source-level debugging.</p>

<p style="text-align:justify;">The second part of the lab focuses on host-device debugging using ROCgdb. You will set breakpoints in both CPU and GPU code, launch the application within the debugger, and step through execution. You will explore GPU execution by selecting wavefronts and lanes, inspecting thread behavior, and analyzing how parallel execution maps to source code. This phase demonstrates how ROCgdb provides unified visibility into both host and device execution.</p>

<p style="text-align:justify;">The final part of the lab focuses on detailed inspection and verification. You will examine registers, memory contents, and source-to-ISA mappings to understand execution behavior at a low level. You will also observe how breakpoints behave in a parallel execution model and verify correct kernel output after debugging. This process enables identification of logical errors, incorrect indexing, and memory-related issues.</p>

<p style="text-align:justify;">Through this hands-on lab, you will gain practical experience with ROCgdb, learn how to debug heterogeneous applications effectively, and develop a deeper understanding of GPU execution. The lab provides a solid foundation for diagnosing issues, analyzing execution behavior, and ensuring correctness in GPU-accelerated applications.</p>

<p style="text-align:justify;">**Note on GPU Architecture Compatibility**: When compiling for debugging, ensure that the `--offload-arch` flag matches your physical GPU architecture. You can identify your GPU architecture using the `rocminfo` command. Common architecture identifiers include gfx1100, gfx1151, and gfx1200 for consumer GPUs, and gfx90a or gfx942 for data center GPUs.</p>

## Step 1: Environment Readiness and Debug Build Setup

<p style="text-align:justify;">The first step in GPU debugging is to verify that your ROCm debugging toolchain is properly installed and your GPU is correctly recognized. You will then compile a HIP application with debug symbols enabled, which is essential for source-level debugging. Debug symbols allow the debugger to map GPU execution back to your source code, enabling meaningful inspection of variables, execution flow, and kernel behavior.</p>

### Verify the Environment

Download a fixed version of the ROCm examples repository, which contains the debugging example application for this lab.

In [ ]:
!curl -fL https://codeload.github.com/ROCm/rocm-examples/tar.gz/dd808c7ad144843aa3f6eaa0f877315141eaa84a -o rocm-examples.tar.gz \
    && mkdir rocm-examples \
    && tar -xzf rocm-examples.tar.gz --strip-components=1 -C rocm-examples

This extracts the fixed ROCm examples source archive into the current directory. The repository contains various HIP programming examples and tools demonstrations, including the ROCgdb debugging example.

Navigate to the ROCgdb example directory, which contains the HIP matrix multiplication application that will be debugged in this lab.

In [ ]:
%cd rocm-examples/Tools/ROCgdb

In [ ]:
!rocminfo

<p style="text-align:justify;">This command confirms that the ROCm runtime is active and displays information about your AMD GPU. Look for the GPU architecture identifier (for example, gfx1100, gfx1151, gfx1200, gfx90a, or gfx942) in the output, as you will need this for the compilation step. The architecture identifier indicates the specific GPU instruction set and capabilities, and matching this identifier during compilation ensures that debug information is correctly generated for your hardware.</p>

Verify that the HIP compiler is installed and accessible. The HIP compiler is required for building GPU applications with debug support.

In [ ]:
!hipcc --version

This prints the installed HIP compiler version, confirming that the HIP toolchain is available for building the application. The version information helps verify that you have a compatible compiler for your ROCm installation.

Verify that ROCgdb is installed and accessible in your environment.

In [ ]:
!rocgdb --version

This confirms that ROCgdb is installed and displays its version information. ROCgdb is built on top of standard GDB and includes AMD-specific extensions for GPU debugging.

### Compile the ROCgdb Example with Debug Support

<p style="text-align:justify;">Compile the HIP application with debug symbols enabled and optimizations disabled using auto-detected GPU architecture. This compilation configuration is critical for effective debugging. The `-g` flag instructs the compiler to include debug symbols in the binary, which map machine code back to source lines. The `-O0` flag disables compiler optimizations, ensuring that the code executes in the same order as written in the source, making stepping and variable inspection more predictable. The script automatically detects your GPU architecture using `rocminfo` and sets the appropriate `--offload-arch` flag.</p>

In [ ]:
# Auto-detect GPU arch and compile with debug flags
import subprocess, re

rocminfo_out = subprocess.check_output(["bash", "-lc", "rocminfo"], text=True)
match = re.search(r"gfx\d+", rocminfo_out)
arch = match.group(0) if match else None
print("Detected GPU arch:", arch)

if not arch:
    raise RuntimeError("Could not detect a gfx arch from rocminfo output.")

cmd = [
    "hipcc",
    "-g", "-O0",
    f"--offload-arch={arch}",
    "-I", "../../Common",
    "-I", "../../External",
    "main.hip",
    "-o", "rocgdb_example",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("Build complete. Binary: ./rocgdb_example")

<p style="text-align:justify;">This compiles the HIP application with debug symbols (`-g`) and disables optimization (`-O0`) to enable accurate source-level debugging on the GPU. The script automatically detects your GPU architecture from `rocminfo` output and uses it for the `--offload-arch` flag. The `-I` flags specify include directories for common headers and external dependencies using relative paths. The output is an executable named `rocgdb_example` that contains both host and device debug information.

Debug builds are typically larger than optimized builds because they include symbol tables, source line mappings, and variable information. They also execute more slowly because optimizations are disabled. However, these trade-offs are essential for effective debugging, as optimized code can reorder instructions, eliminate variables, and inline functions in ways that make debugging difficult or impossible.</p>

## Step 2: Host-Device Debugging Using ROCgdb

With the application compiled with debug symbols, you will now launch ROCgdb and debug both CPU and GPU execution in a single session. This unified debugging experience allows you to set breakpoints in host code, step into kernel launches, inspect GPU execution at the wavefront and lane level, and examine the relationship between CPU orchestration and GPU computation.

### Launch ROCgdb in a Terminal

ROCgdb is an interactive debugger that requires you to type commands at a `(gdb)` prompt. It cannot be run effectively from Jupyter code cells because it needs interactive input/output. Instead, you will open a terminal within JupyterLab and run ROCgdb there.

**To start ROCgdb:**

1. In JupyterLab, click **File → New → Terminal** to open a terminal window
2. Navigate to the ROCgdb example directory:
   ```bash
   cd rocm-examples/Tools/ROCgdb
   ```
   If your workspace is in a different location, adjust the path accordingly. For example, if you're in a workspace called "CUDA-to-HIP Migration Labs", use:
   ```bash
   cd "CUDA-to-HIP Migration Labs/rocm-examples/Tools/ROCgdb"
   ```
3. Launch ROCgdb with the compiled executable:
   ```bash
   rocgdb ./rocgdb_example
   ```

This will start the ROCgdb debugger and display the `(gdb)` prompt. Debug symbols are read and the debugger is ready to accept commands. Keep this terminal window open throughout the lab.

**All subsequent debugging commands in this lab should be typed in the terminal at the `(gdb)` prompt, not in Jupyter cells.** The following sections provide complete documentation of the debugging workflow, with code blocks showing the exact commands to type.

**Note**: The cell above will launch ROCgdb in interactive mode. For this lab, the remaining debugging commands should be entered directly in the ROCgdb prompt that appears, not as Jupyter cells. The following sections provide markdown documentation of the debugging workflow, with code blocks showing the commands to type at the `(gdb)` prompt.

### Debug Host Code

Set a breakpoint at the `main()` function to pause execution at the start of the program. Breakpoints are markers that tell the debugger where to pause execution, allowing you to inspect program state at specific locations.

```gdb
(gdb) break main
```

This sets a host-side breakpoint, causing execution to pause at the start of the program when you run it. ROCgdb confirms that the breakpoint was set and shows the source location.

Run the application under the debugger. The `run` command starts program execution from the beginning.

```gdb
(gdb) run
```

The application begins execution and stops immediately at the `main()` breakpoint. ROCgdb displays the source line where execution paused, allowing you to inspect the program state before any GPU operations occur.

Use the `list` command to scroll through the source file and identify the exact line where the HIP kernel is launched. The `list` command displays source code in blocks of 10 lines at a time, helping you navigate the program.

```gdb
(gdb) list
```

**Note**: The `list` command displays the file in blocks of 10 lines at a time. As the HIP kernel launch statement may not appear in the initial output, continue entering the `list` command repeatedly until the full HIP kernel launch line is visible. The kernel launch typically appears as a line containing `<<<grid_dim, block_dim, ...>>>`, which is HIP's syntax for specifying kernel execution configuration.

Once you locate the kernel launch line (for example, around line 194 in the example), set a breakpoint at that line to pause execution just before the GPU kernel is launched.

```gdb
(gdb) break main.hip:194
```

This pauses execution just before the GPU kernel is launched, allowing you to inspect the state of host variables and arguments before they are passed to the device.

Continue execution until the kernel launch breakpoint is reached.

```gdb
(gdb) continue
```

Execution resumes from the `main()` breakpoint and continues until it reaches the kernel launch breakpoint. ROCgdb displays the source line showing the kernel launch configuration, confirming that you are about to dispatch work to the GPU.

### Debug GPU Kernel Execution

Set a breakpoint inside the GPU kernel to pause execution when wavefronts reach that point. Setting a device breakpoint is similar to setting a host breakpoint, but it applies to GPU code. Since GPU code is loaded dynamically at kernel dispatch time, ROCgdb initially marks device breakpoints as "pending" and activates them when the kernel is actually loaded.

```gdb
(gdb) break main.hip:65
```

In the fixed source, line 65 is executable and follows initialization of `tx`, `ty`, `bx`, and `by`.

**Note**: Since GPU code is loaded at runtime, ROCgdb marks this as a pending device breakpoint. The debugger will ask you to confirm that you want to set a pending breakpoint. Type `y` and press Enter to accept the pending device breakpoint.

```gdb
(gdb) y
```

This instructs ROCgdb to activate the breakpoint when the GPU kernel is loaded at dispatch. The kernel code is loaded only at dispatch time, so this breakpoint is initially marked as pending and will become active when the first wavefront begins execution.

Continue execution to enter the kernel and trigger the device breakpoint.

```gdb
(gdb) continue
```

Execution stops inside the GPU kernel when a wavefront reaches the breakpoint. ROCgdb displays information about the stopped wavefront, including its work group coordinates and the source line where it stopped. This confirms that GPU execution is now paused and you can begin inspecting device state.

### Inspect GPU Waves

List all CPU threads and GPU wavefronts to see the complete execution context. ROCgdb presents both CPU threads and GPU wavefronts in a unified thread view, making it easy to understand the overall execution state.

```gdb
(gdb) info threads
```

<p style="text-align:justify;">This displays CPU threads and GPU wavefronts, with GPU waves shown as AMDGPU threads. The first entries correspond to host threads that manage runtime operations such as kernel launches and synchronization. Entries labeled "AMDGPU Wave" represent GPU wavefronts, the basic execution units on AMD GPUs. Each wavefront entry shows details like work group IDs and the current source line.</p>

<p style="text-align:justify;">In this case, all wavefronts stop at the same line (main.hip:65), indicating they hit a device breakpoint. ROCgdb presents both CPU and GPU execution in a unified thread view, allowing consistent debugging with standard GDB commands. Press `c` and Enter to continue without paging if the output is long.</p>

Select a specific GPU wavefront to focus your debugging efforts. The `thread` command switches the debugging context to a particular execution thread, which can be either a CPU thread or a GPU wavefront.

```gdb
(gdb) thread <wave-id>
```

**Note**: Replace `<wave-id>` with the ID of any AMDGPU Wave from the `info threads` list. Use `info threads` to find this ID—ROCgdb lists all execution contexts, and the leftmost column shows the thread IDs used with the `thread` command.

<p style="text-align:justify;">This focuses debugging on a single wavefront to avoid switching between multiple GPU execution contexts. The output shows that ROCgdb has switched to the selected GPU wavefront and is focused on its default lane (lane 0). The identifier reflects the GPU hierarchy, including work group coordinates and lane index. Execution is currently inside the kernel `matrix_multiplication_kernel<16u>`, stopped at line 65 (main.hip), where the device breakpoint was hit. This confirms that GPU execution is correctly mapped to source code and enables lane-specific debugging, including stepping, register inspection, and memory analysis.</p>

### Inspect GPU Lanes

Display information about all lanes in the selected wavefront. Each wavefront consists of multiple lanes (work items) that execute in lockstep, and the `info lanes` command shows the state of each lane.

```gdb
(gdb) info lanes
```

<p style="text-align:justify;">This shows per-lane execution state and activity within the selected wavefront. The `info lanes` command shows all lanes in the selected GPU wavefront. All lanes are active, indicating no divergence. Each lane is executing `matrix_multiplication_kernel<16u>` and is stopped at main.hip:65, where the breakpoint was hit. This confirms lockstep execution and allows lane-level inspection for debugging.</p>

Select a specific lane to examine its state in detail. Just as you can switch between wavefronts using the `thread` command, you can switch between lanes using the `lane` command.

```gdb
(gdb) lane 5
```

**Note**: You can enter any lane ID as needed. This example uses lane 5, but you can select any lane from 0 to the maximum lane count for your GPU architecture.

<p style="text-align:justify;">This selects an individual work item so that subsequent register and variable inspections reflect that specific thread's execution context. ROCgdb confirms that it has switched to lane 5 in the selected GPU wavefront. Subsequent inspections now reflect this lane's state. Execution remains in `matrix_multiplication_kernel<16u>` at main.hip:65, where the breakpoint was hit, enabling lane-specific debugging of variables, registers, and memory.</p>

### Step Through Kernel Code

Step over a source line to advance execution to the next line in the kernel. The `next` command executes the current line and stops at the next source line, similar to standard GDB.

```gdb
(gdb) next
```

<p style="text-align:justify;">This advances execution to the next source line in the kernel for the active lanes. However, you may observe that `next` does not advance execution in the current wave. Instead, ROCgdb switches to another wavefront that hit the same breakpoint at main.hip:65. This occurs because the breakpoint remains active and is triggered by multiple wavefronts executing in parallel. The debugger resets to lane 0 of the new wave, with all lanes stopped at the breakpoint. To step within a single wave, the device breakpoint must be disabled (which you will do later in this lab).</p>

If you want to step into function calls, including device or runtime helpers, use the `step` command.

```gdb
(gdb) step
```

This enters function calls, including device or runtime helpers, if present. The `step` command is useful when you want to examine the implementation of called functions.

If you step into a helper function and want to return to the kernel source, use the `finish` command.

```gdb
(gdb) finish
```

<p style="text-align:justify;">This runs until the current function returns, bringing execution back to the kernel source. However, you may receive a message indicating that `finish` cannot be used because execution is already in the top-level kernel function (matrix_multiplication_kernel). The `finish` command only works within nested function calls, where there is a caller to return to. This behavior is expected. To proceed, use `next`, `continue`, or disable the breakpoint to advance execution.</p>

### Inspect Source and ISA Mapping

View the mixed source and instruction set architecture (ISA) to see how your HIP source code maps to actual GPU instructions. This advanced debugging technique helps you understand how the compiler translates high-level code into GPU operations.

```gdb
(gdb) disassemble /m
```

This displays HIP source lines alongside the underlying AMDGPU instructions, helping analyze control flow and divergence. Press `c` and Enter to continue without paging if the output is long.

<p style="text-align:justify;">The `disassemble /m` command shows a mixed view of source code and corresponding AMDGPU ISA instructions for the kernel `matrix_multiplication_kernel<16u>`. It lists instruction addresses alongside scalar (s_) and vector (v_) operations, which handle shared and per-lane data. The output also includes memory and control flow instructions generated by the compiler. The current source line (main.hip:65) is mapped to these instructions via debug information, allowing you to see how high-level code translates into GPU operations. This view is useful for analyzing execution behavior at the instruction level.</p>

### Inspect Registers

Display the hardware register values for the selected wavefront and lane. Registers are the fastest storage locations on the GPU and are used for computation, addressing, and control flow. Inspecting registers provides low-level insight into GPU execution.

```gdb
(gdb) info registers
```

This displays scalar and vector register values along with execution state such as the program counter. Press `c` and Enter to continue without paging if the output is long.

<p style="text-align:justify;">The `info registers` command displays the hardware register state for the selected wavefront and lane. It includes scalar registers (SGPRs), which are shared across the wave, and vector registers (VGPRs), which hold lane-specific data. Special registers like the program counter and execution mask indicate the current instruction and active lanes. This output enables low-level inspection of GPU execution, useful for analyzing computations, memory access, and divergence.</p>

### Inspect Variables and Indices

Inspect thread indices to understand which work item is currently being debugged. Thread indices (block and thread coordinates) are fundamental to understanding GPU execution, as they determine which portion of data each work item processes.

```gdb
(gdb) print bx
(gdb) print by
(gdb) print tx
(gdb) print ty
```

<p style="text-align:justify;">These commands display block and thread indices used by the kernel for computation. The values of `bx`, `by`, `tx`, and `ty` are all zero because the debugger is focused on lane 0 in work group (0,0). This corresponds to the first thread in the kernel, confirming that index values are reported correctly. If you switched to a different lane or wavefront earlier, you would see different index values reflecting that thread's position in the execution grid.</p>

### Inspect Device Memory

Print device pointers to verify that kernel arguments are correctly passed. Device pointers are memory addresses in GPU memory space, and verifying them ensures that the kernel received valid input and output buffers.

```gdb
(gdb) print A
(gdb) print B
(gdb) print C
```

<p style="text-align:justify;">These commands show the GPU memory addresses passed to the kernel. The output displays the device memory addresses of kernel arguments. `A` and `B` are read-only (`const float *`), while `C` is writable (`float *`). The hexadecimal values are base addresses of GPU buffers. This confirms valid device pointers and correct argument setup.</p>

Dump memory contents to verify input and output data. The `x` command (examine memory) allows you to view the actual data stored in GPU memory.

```gdb
(gdb) x/16wf A
(gdb) x/16wf B
(gdb) x/16wf C
```

These commands display the contents of GPU memory to verify input and output data. The `x/16wf` command displays 16 floating-point values from GPU memory at the addresses of A, B, and C. The format specifiers mean: 16 values, word (4-byte) size, floating-point format.

<p style="text-align:justify;">Matrices A and B show consistent values, confirming correct initialization in device memory. Matrix C is an output buffer allocated with `hipMalloc` and has not yet been written at this breakpoint, so its initial contents are unspecified. This inspection verifies input initialization and pointer setup; the program's final numerical check verifies the output after the kernel completes.</p>

## Step 3: Kernel Execution, Verification and Completion

After inspecting GPU execution state, registers, and memory at the breakpoint, you will now disable the kernel breakpoint and allow the application to complete execution normally. This final step verifies that the kernel produces correct results when executed without debugging interruptions.

### Disable Device Breakpoint

List all active breakpoints to identify the device breakpoint that needs to be disabled.

```gdb
(gdb) info breakpoints
```

The `info breakpoints` command lists all active host and device breakpoints. It shows their IDs, locations, and hit counts. In this case, two host breakpoints in `main()` and at the kernel launch have each been hit once, while the device breakpoint in `matrix_multiplication_kernel<16u>` at main.hip:65 has been hit multiple times by different wavefronts. This confirms correct breakpoint tracking and highlights repeated hits due to parallel GPU execution.

Note the breakpoint number for the device breakpoint (typically breakpoint 3), then disable it to prevent repeated stops by multiple GPU wavefronts.

```gdb
(gdb) disable 3
```

**Note**: Replace `3` with the actual breakpoint number from your `info breakpoints` output. Disabling the device breakpoint prevents repeated stops by multiple GPU wavefronts, allowing the kernel to execute to completion.

After disabling the breakpoint, the kernel runs without interruption when you continue execution.

### Resume Execution and View Output

Continue execution to completion, allowing all remaining wavefronts to finish their work.

```gdb
(gdb) continue
```

The program finishes execution and prints the final output message. After you disabled the device breakpoint, the kernel runs without interruption. The message **"Validation passed."** confirms correct GPU execution. The remaining messages indicate normal shutdown, including unloading GPU code and termination of threads, with the program exiting successfully. This validates that the matrix multiplication kernel produced correct results.

### Exit ROCgdb

Quit the debugger to end the debugging session.

```gdb
(gdb) quit
```

This ends the ROCgdb session and returns control to the shell.

## Summary

<p style="text-align:justify;">In this lab, you prepared a HIP application for debugging by compiling it with debug symbols and disabling optimizations. You used ROCgdb to debug both host and GPU execution in a unified session, setting breakpoints in CPU code and GPU kernels. You inspected wavefront and lane behavior to understand parallel execution patterns, examining how multiple work items execute simultaneously. You examined registers and memory to verify kernel inputs, outputs, and intermediate computations at a low level. Finally, you allowed the program to complete execution with validated results, confirming that the kernel produced correct output.</p>

<p style="text-align:justify;">This workflow demonstrates a complete source-level debugging process for GPU applications using ROCgdb. The techniques you learned—setting breakpoints in device code, inspecting wavefronts and lanes, examining registers and memory, and correlating execution with source code—form the foundation for debugging real-world GPU applications. ROCgdb's unified CPU-GPU debugging environment enables you to diagnose issues that span both host and device code, making it an essential tool for GPU application development.</p>

<p style="text-align:justify;">Key takeaways from this lab include understanding that GPU debugging requires debug symbols (`-g`) and disabled optimizations (`-O0`), recognizing that device breakpoints are pending until kernel dispatch, knowing that wavefronts execute in parallel and may hit breakpoints multiple times, and learning that lane-level inspection provides detailed insight into individual work items. These concepts and techniques will serve you well as you develop and debug more complex GPU-accelerated applications.</p>